In [ ]:
# run this notebook after 16_calculate_af_python
# use this notebook to determine distance filter using quads, apply distance filter to sibling differences, 
# determine af filter using quads, apply af filter to sibling differences,
# remove outlier siblings (as described in methods), and write final counts per trio and filter-passing sibling pair
# we need ukb distances between adjacent mutations (./ukb/trios_distances.txt)
# we need allele frequencies of ukb quad differences in (./ukb/insibs_intrios_acs.txt, 
# ./ukb/insibs_notintrios_acs.txt, ./ukb/notinsibs_intrios_acs.txt)
# we need ukb ibd2 after trim and qc (./ukb/sibs_ibd2.csv)
# after this, run 18_annotate_trio_sibling_dnm_COSMIC_python

In [ ]:
library(dplyr)
library(data.table)
library(ggplot2)
library(tidyr)
library(stringr)

In [ ]:
quad = fread("./relatedness_dec20/quads_new.txt", sep = "\t")
flagged = fread("./flagged_samples.tsv", sep = "\t", header= TRUE)
quad = quad %>% filter(!(off1 %in% flagged$s) & !(off2 %in% flagged$s) & !(par1 %in% flagged$s) & !(par2 %in% flagged$s) )
trio_difs = fread("./trios_snps_after_standard_QC_GIAB.txt", sep = "\t")
sib_difs = fread("./sibs_snps_QC_GIAB.txt")
sib_difs$pair = paste(sib_difs$idv1, sib_difs$idv2, sep = "_")
quad$pair = paste(quad$off1, quad$off2, sep = "_")
sib_difs_quad = sib_difs %>% filter(pair %in% quad$pair)
sib_difs_quad$idv = ifelse(sib_difs_quad$idv1_gt %in% c("0/0", "0|0"), 
                          sib_difs_quad$idv2, sib_difs_quad$idv1)
length(unique(sib_difs_quad$pair))
sib_difs_quad = sib_difs_quad %>% select(chr, pos, alleles, idv)
sib_difs_quad = unique(sib_difs_quad)

In [ ]:
remove_dense_windows = function(positions, window_size = 100000, min_mutations = 3) {
  positions = sort(positions)
  to_remove = rep(FALSE, length(positions))
  
  # use a sliding index-based window rather than 1-bp step for efficiency
  start_idx = 1
  for (end_idx in 1:length(positions)) {
    while (positions[end_idx] - positions[start_idx] > window_size) {
      start_idx = start_idx + 1
    }
    if (end_idx - start_idx + 1 >= min_mutations) {
      to_remove[start_idx:end_idx] = TRUE
    }
  }
  return(to_remove)
}

apply_distance_filter = function(pos_file, ws){
    pos_file$distance_filter = NA
    idvs = unique(pos_file$idv)
    for (i in idvs){
        ind_idv = which(pos_file$idv == i)
        ind_pos = pos_file[ind_idv,]
        for (c in 1:22){
            ind_chr_idx = which(ind_pos$chrom == c)
            ind_chr = ind_pos[ind_chr_idx,]
            if (nrow(ind_chr) != 0){
                remove_chr = remove_dense_windows(ind_chr$start, window_size = ws, min_mutations = 3)
                idx = ind_idv[ind_chr_idx]
                pos_file$distance_filter[idx] = remove_chr
            }
        } 
    }
    return(pos_file)
}

calculate_distances = function(positions){
    positions = sort(positions, decreasing = FALSE)
    d = rep(NA, length(positions))
    for (i in 1:(length(positions))){
        if (i == 1){
            d[i] = positions[i+1] - positions[i] 
        } else if (i == length(positions)){
            d[i] = abs(positions[i] - positions[i-1])
        } else {
            d[i] = min(c(abs(positions[i] - positions[i-1]), abs(positions[i+1]-positions[i])))
        }
    }
    return(d)
}

In [ ]:
quad$tot_after_dist = NA
quad$tot_after_dist_in_trio = NA
m = 3 
ws = c(0, 100, 1000, 10000, 100000, 1000000, 10000000)
sib_difs_quad = data.frame("chrom" = NA,
                           "start" = NA, 
                           "end" = NA, 
                           "idv" = NA, 
                           idv1_gt = NA,
                           idv2_gt = NA,
                           idv1 = NA,
                           idv2= NA,
                           chr = NA,
                           pos = NA,
                           filter_1mb_3 = NA,
                          in_trio = NA)
result = data.frame(ws, proportion = NA, total = NA)
for (j in 1:length(ws)){
    w = ws[j]
    for (i in 1:nrow(quad)){
        i1 = quad$off1[i]
        i2 = quad$off2[i]
        idvs = c(i1, i2)
        # subset sib file to those individuals 
        sib_pair = sib_difs %>% filter(idv1 == i1 & idv2 == i2)
        # turn sib file to bed file 
        sib_bed = data.frame(chrom = sib_pair$chr, start = sib_pair$pos,
                        end = sib_pair$pos+1)
        sib_bed$idv = sib_pair$idv
        sib_bed$idv1_gt = sib_pair$idv1_gt
        sib_bed$idv2_gt = sib_pair$idv2_gt
        sib_bed$idv1 = sib_pair$idv1
        sib_bed$idv2 = sib_pair$idv2
        sib_bed$chr = sib_pair$chr
        sib_bed$pos = sib_pair$pos
        sib_ibd = apply_distance_filter(sib_bed, w)
        sib_ibd = sib_ibd %>% filter(!distance_filter)
        quad$tot_after_dist[i] = nrow(sib_ibd)
        # count the number of sib difs that are also trio difs 
        trio_pair = trio_difs %>% filter(off %in% idvs)
        # turn trio file to bed file
        trio_bed = data.frame(chrom = trio_difs$chr, start = trio_difs$pos, end = trio_difs$pos+1)
        trio_bed$in_trio = TRUE
        sib_in_trio = merge(sib_ibd, trio_bed, by = c("chrom", "start", "end"), all.x = TRUE)
        if (w == 0){
            sib_difs_quad = rbind(sib_difs_quad, sib_in_trio)
        }
        quad$tot_after_dist_in_trio[i] = sum(!is.na(sib_in_trio$in_trio))
    }
    result$proportion[j] =sum(quad$tot_after_dist_in_trio)/sum(quad$tot_after_dist)
    result$total[j] = sum(quad$tot_after_dist)
}

In [ ]:
result_plot = result %>% select(ws, proportion)
result_plot$dataset = "aou"
result_plot_ukb = result_plot
result_plot_ukb$proportion = c(0.5389, 0.5415,0.6067, 0.7059, 0.7518, 0.7581, 0.7659)
result_plot_ukb$dataset = "ukb"
result_plot = rbind(result_plot, result_plot_ukb)
ggplot(result_plot, aes(x = log10(ws+1), y = proportion, color = dataset)) + geom_point(size = 3) + theme_bw(base_size = 12) +
labs(x = "log10(window size)", y = "Proportion of sib differences in trio") 

In [ ]:
# distance to nearest mutation for all mutations 
sib_difs_quad = sib_difs_quad[2:nrow(sib_difs_quad),]
sib_difs_quad = sib_difs_quad %>% select(chr, pos, idv1, idv2, idv, filter_1mb_3, in_trio)
nrow(sib_difs_quad)
sib_difs_quad = unique(sib_difs_quad)
nrow(sib_difs_quad)
sib_difs_quad_in_trio = sib_difs_quad %>% filter(in_trio)
sib_difs_quad_not_in_trio = sib_difs_quad %>% filter(is.na(in_trio))

In [ ]:
# distance to nearest mutation of de novo mutations in trio 
aou_distances = rep(NA, nrow(sib_difs_quad_in_trio))
counter = 1
for (i in 1:nrow(quad)){
    i1 = quad$off1[i]
    i2 = quad$off2[i]
    idvs = c(i1, i2)
    pair_idx = which(sib_difs_quad_in_trio$idv1 == i1 & sib_difs_quad_in_trio$idv2 == i2)
    pair_giab = sib_difs_quad_in_trio[pair_idx,]
    # now, for each individual, apply filter 
    for (individual in c(i1, i2)){
        ind_idx = which(pair_giab$idv == individual)
        ind_giab = pair_giab[ind_idx,]
        # go across chromosomes 
        for (c in 1:22){
            ind_chr_idx = which(ind_giab$chr == c)
            ind_chr = ind_giab[ind_chr_idx,]
            if (nrow(ind_chr) > 1){
                distances = calculate_distances(ind_chr$pos)
                aou_distances[counter:(counter+length(distances)-1)] = distances
                counter = counter + length(distances)
            }
        }
    }
}
aou_distances_in_trio = aou_distances

In [ ]:
# distance to nearest mutation of de novo mutations not in trio 
aou_distances = rep(NA, nrow(sib_difs_quad_not_in_trio))
counter = 1
for (i in 1:nrow(quad)){
    i1 = quad$off1[i]
    i2 = quad$off2[i]
    idvs = c(i1, i2)
    pair_idx = which(sib_difs_quad_not_in_trio$idv1 == i1 & sib_difs_quad_not_in_trio$idv2 == i2)
    pair_giab = sib_difs_quad_not_in_trio[pair_idx,]
    # now, for each individual, apply filter 
    for (individual in c(i1, i2)){
        ind_idx = which(pair_giab$idv == individual)
        ind_giab = pair_giab[ind_idx,]
        # go across chromosomes 
        for (c in 1:22){
            ind_chr_idx = which(ind_giab$chr == c)
            ind_chr = ind_giab[ind_chr_idx,]
            if (nrow(ind_chr) > 1){
                distances = calculate_distances(ind_chr$pos)
                aou_distances[counter:(counter+length(distances)-1)] = distances
                counter = counter + length(distances)
            }
        }
    }
}
aou_distances_not_in_trio = aou_distances

In [ ]:
aou_distances_in_trio = aou_distances_in_trio[which(!is.na(aou_distances_in_trio))]
aou_distances_not_in_trio = aou_distances_not_in_trio[which(!is.na(aou_distances_not_in_trio))]

df = rbind(
  data.frame(distances = aou_distances_in_trio, group = "AoU sib differences in trio"),
  data.frame(distances = aou_distances_not_in_trio, group = "AoU sib differences not in trio")
)

df$distances[which(df$distances == 0)] = 1

ggplot(df, aes(x = distances, color = group)) +
  stat_ecdf(geom = "step", linewidth = 1) +
  labs(title = "", x = "Distance log10(bp)", y = "Percentile",
       color="Distances between adjacent\nmutations after GIAB filtering") +
  theme_bw(base_size = 12)  + scale_x_log10(                              # x-range after log10
    breaks = c(1e2, 1e4, 1e6, 1e8)) 

In [ ]:
# read aou distances to nearest mutations (trios)
trios = fread("./trios_snps_after_standard_QC_GIAB.txt")
trio_distances = rep(NA, nrow(trios))
counter = 1
trios$proband = paste(trios$par1, trios$par2, trios$off, sep = "_")
proband_list = unique(trios$proband)
for (i in 1:length(proband_list)){
    p = proband_list[i]
    proband_muts = trios %>% filter(proband == p)
    for (c in 1:22){
        proband_muts_chr = proband_muts %>% filter(chr == c)
        if (nrow(proband_muts_chr) > 1){
            distances = calculate_distances(proband_muts_chr$pos)
            trio_distances[counter:(counter+length(distances)-1)] = distances
            counter = counter + length(distances)
        }
    }
}

trio_distances = trio_distances[1:(counter-1)]

In [ ]:
# read ukb trios 
ukb_trios_distances = fread("./ukb/trios_distances.txt")
df = rbind(
  data.frame(distances = trio_distances, group = "aou trio"),
  data.frame(distances = ukb_trios_distances$V1, group = "ukb trio")  
)

df$distances[which(df$distances == 0)] = 1

ggplot(df, aes(x = distances, color = group)) +
  stat_ecdf(geom = "step", linewidth = 1) +
  labs(title = "", x = "Distance log10(bp)", y = "Percentile",
       color="Distances between adjacent\nmutations after GIAB filtering") +
  theme_bw(base_size = 12)  + scale_x_log10(                              # x-range after log10
    breaks = c(1e2, 1e4, 1e6, 1e8)) 

In [ ]:
# apply distance filter to sibling differences 
sibs_all = fread("./relatedness_dec20/sibs_final.txt")
sib_difs$distance_filter = NA
ws = 100000
minmuts = 3
for (i in 1:nrow(sibs_all)){
    message(i)
    i1 = sibs_all$idv1[i]
    i2 = sibs_all$idv2[i]
    pair_idx = which(sib_difs$idv1 == i1 & sib_difs$idv2 == i2)
    pair_giab = sib_difs[pair_idx,]
    # now, for each individual, apply filter 
    for (individual in c(i1, i2)){
        ind_idx = which(pair_giab$idv == individual)
        ind_giab = pair_giab[ind_idx,]
        # go across chromosomes 
        for (c in 1:22){
            ind_chr_idx = which(ind_giab$chr == c)
            ind_chr = ind_giab[ind_chr_idx,]
            if (nrow(ind_chr) != 0){
                remove_chr = remove_dense_windows(ind_chr$pos, window_size = ws, min_mutations = minmuts)
                idx_snps_filtered = pair_idx[ind_idx[ind_chr_idx]]
                sib_difs$distance_filter[idx_snps_filtered] = remove_chr
            }
        }
    }
}

sib_difs = sib_difs %>% filter(!distance_filter)
fwrite(sib_difs, "./sibs_snps_QC_GIAB_distance.txt", sep = "\t", quote = FALSE, row.names = FALSE)

In [ ]:
# now read af 
af = fread("./all_difs_allele_frequencies_feb15.tsv", sep = "\t", header=TRUE)
names(af) = c("locus", "alleles","vid","ac", "an", "af")
af = af %>%
  separate(locus, into = c("chr", "pos"), sep = ":", remove = FALSE) %>%
  mutate(
    chr = str_remove(chr, "^chr"),
    pos = as.integer(pos),
    ref = str_match(alleles, '"([^"]+)"\\s*,\\s*"([^"]+)"')[, 2],
    alt = str_match(alleles, '"([^"]+)"\\s*,\\s*"([^"]+)"')[, 3]
  )
af = af %>% filter(chr %in% 1:22)
af$chr = as.numeric(af$chr)
af$pos = as.numeric(af$pos)

In [ ]:
sib_difs_af = merge(sib_difs, af, by = c("chr", "pos", "ref", "alt"), all.x = TRUE)
trio_difs = trio_difs %>% filter(chr %in% 1:22)
trio_difs$chr = as.numeric(trio_difs$chr)
trio_difs_af = merge(trio_difs, af, by = c("chr", "pos", "ref", "alt"), all.x = TRUE)
sibs = sibs_all 


In [ ]:
# first, annotate sib_in_trio and sib_not_in_trio afs
ibd2_0.75cm = fread("./ibd2_all_trimmed_0.75cM_jan12.txt")
ibd2 = ibd2_0.75cm %>% filter(length_cm >= 6)
ibd2 = unique(ibd2)

sib_in_trio = c()
sib_not_in_trio = c()
trio_in_sib = c()
trio_not_in_sib = c()
quad$tot_after_dist = NA
quad$tot_after_dist_in_trio = NA
for (i in 1:nrow(quad)){
    i1 = quad$off1[i]
    i2 = quad$off2[i]
    idvs = c(i1, i2)
    ibd2_pair = ibd2 %>% filter(ID1 == i1 & ID2 == i2)
    ibd2_pair_bed = data.frame(chrom = ibd2_pair$Chr, start = ibd2_pair$new_start_bp, end = ibd2_pair$new_end_bp)
    sib_idv = sib_difs_af %>% filter(idv1 == i1 & idv2 == i2)
    trio_idv = trio_difs_af %>% filter(off %in% idvs)
    sib_sub = sib_idv[,c("chr", "pos", "ref", "alt", "af")]
    trio_sub = trio_idv[,c("chr", "pos", "ref", "alt", "af")]
    trio_sub$in_trio = TRUE
    merged = merge(sib_sub, trio_sub, c("chr", "pos", "ref", "alt", "af"), all.x = TRUE)
    merged = unique(merged)
    quad$tot_after_dist[i] = nrow(sib_idv)
    sib_in_trio = c(sib_in_trio, merged$af[which(merged$in_trio)])
    sib_not_in_trio = c(sib_not_in_trio, merged$af[which(is.na(merged$in_trio))])
    quad$tot_after_dist_in_trio[i] = sum(!is.na(merged$in_trio))
    # next, subset trio to ibd2 segments pertaining to these two idvs 
    trio_bed = data.frame(chrom = trio_sub$chr, start = trio_sub$pos, end = trio_sub$pos+1)
    merged2 = bedtoolsr::bt.intersect(a = trio_bed, b = ibd2_pair_bed, c = TRUE)
    trio_sub$in_ibd = merged2$V4
    trio_sub = trio_sub %>% filter(in_ibd == 1)
    sib_sub$in_sib = TRUE
    merged3 = merge(trio_sub, sib_sub, c("chr", "pos", "ref", "alt", "af"), all.x = TRUE)
    merged3 = unique(merged3)
    trio_in_sib = c(trio_in_sib, merged3$af[which(merged3$in_sib)])
    trio_not_in_sib = c(trio_not_in_sib, merged3$af[which(is.na(merged3$in_sib))])
}

aou_results = data.frame(af = c(sib_in_trio, sib_not_in_trio, trio_not_in_sib),
                         category= c(rep("in_sib_in_trio", length(sib_in_trio)),
                                    rep("in_sib_not_trio", length(sib_not_in_trio)),
                                    rep("in_trio_not_sib", length(trio_not_in_sib))))
aou_results$src = rep("AoU", nrow(aou_results))

# ukb results 
in_sib_in_trio = fread("./ukb/insibs_intrios_acs.txt", header=TRUE)

in_sib_in_trio_ukb = in_sib_in_trio$AC/in_sib_in_trio$OBS

in_sib_not_trio = fread("./ukb/insibs_notintrios_acs.txt", header=TRUE)
in_sib_not_trio_ukb = in_sib_not_trio$AC/in_sib_not_trio$OBS

in_trio_not_sib = fread("./ukb/notinsibs_intrios_acs.txt", header=TRUE)
in_trio_not_sib_ukb = in_trio_not_sib$AC/in_trio_not_sib$OBS

ukb_results = data.frame(af = c(in_sib_in_trio_ukb, in_sib_not_trio_ukb, in_trio_not_sib_ukb),
                         category= c(rep("in_sib_in_trio", length(in_sib_in_trio_ukb)),
                                    rep("in_sib_not_trio", length(in_sib_not_trio_ukb)),
                                    rep("in_trio_not_sib", length(in_trio_not_sib_ukb))))
ukb_results$src = rep("UKB", nrow(ukb_results))

ggplot(aou_results, aes(x = af, color = category)) +
  stat_ecdf(geom = "step", linewidth = 1) +
  theme_bw(base_size = 12)  + scale_x_continuous(breaks = seq(0, 1, 0.2), limits = c(0, 1))

ggplot(ukb_results, aes(x = af, color = category)) +
  stat_ecdf(geom = "step", linewidth = 1) +
  theme_bw(base_size = 12)  + scale_x_continuous(breaks = seq(0, 1, 0.2), limits = c(0, 1))

In [ ]:
trial_af_filter = function(sibdifs, triodifs, afcutoff){
    sibdifs = sibdifs %>% filter(af <= afcutoff)
    triodifs = triodifs %>% filter(af <= afcutoff)
    sib_in_trio = c()
    sib_not_in_trio = c()
    trio_in_sib = c()
    trio_not_in_sib = c()
    for (i in 1:nrow(quad)){
        i1 = quad$off1[i]
        i2 = quad$off2[i]
        idvs = c(i1, i2)
        ibd2_pair = ibd2 %>% filter(ID1 == i1 & ID2 == i2)
        ibd2_pair_bed = data.frame(chrom = ibd2_pair$Chr, start = ibd2_pair$new_start_bp, end = ibd2_pair$new_end_bp)
        sib_idv = sibdifs %>% filter(idv1 == i1 & idv2 == i2)
        trio_idv = triodifs %>% filter(off %in% idvs)
        sib_sub = sib_idv[,c("chr", "pos", "ref", "alt", "af")]
        trio_sub = trio_idv[,c("chr", "pos", "ref", "alt", "af")]
        trio_sub$in_trio = TRUE
        merged = merge(sib_sub, trio_sub, c("chr", "pos", "ref", "alt", "af"), all.x = TRUE)
        sib_in_trio = c(sib_in_trio, merged$af[which(merged$in_trio)])
        sib_not_in_trio = c(sib_not_in_trio, merged$af[which(is.na(merged$in_trio))])
        # next, subset trio to ibd2 segments pertaining to these two idvs 
        trio_bed = data.frame(chrom = trio_sub$chr, start = trio_sub$pos, end = trio_sub$pos+1)
        merged = bedtoolsr::bt.intersect(a = trio_bed, b = ibd2_pair_bed, c = TRUE)
        trio_sub$in_ibd = merged[,ncol(merged)]
        trio_sub = trio_sub %>% filter(in_ibd == 1 )
        sib_sub$in_sib = TRUE
        merged = merge(trio_sub, sib_sub, c("chr", "pos", "ref", "alt", "af"), all.x = TRUE)
        trio_in_sib = c(trio_in_sib, merged$af[which(merged$in_sib)])
        trio_not_in_sib = c(trio_not_in_sib, merged$af[which(is.na(merged$in_sib))])
    }
    prop_sib_in_trio = length(sib_in_trio)/(length(sib_in_trio) + length(sib_not_in_trio))
    prop_trio_in_sib = length(trio_in_sib)/(length(trio_in_sib) + length(trio_not_in_sib))
    res = data.frame(category = c("prop_sib_in_trio",
                                 "prop_trio_in_sib"),
                    result = c(prop_sib_in_trio,prop_trio_in_sib))
    res$afcutoff = afcutoff
    return(res)
}

In [ ]:
r = trial_af_filter(sib_difs_af, trio_difs_af, 0.00001)
for (i in c(0.0001, 0.001, 0.01, 0.1,1)){
    message(i)
    r_add = trial_af_filter(sib_difs_af, trio_difs_af, i)
    r = rbind(r, r_add)
}

# plot aou results 
ggplot(data = r , aes(x = afcutoff, y = result, color = category)) + 
geom_point(size=3, shape=17) + theme_bw(base_size=12) + 
scale_x_log10(breaks = c(0.00001,0.0001, 0.001, 0.01, 0.1,1)) + scale_y_continuous(breaks = seq(0.75, 1, by = 0.05), limits = c(0.75, 1))

In [ ]:
ukb_overlaps = fread("./ukb/ukb_af_overlaps.csv", sep = ",")
names(ukb_overlaps) = c("afcutoff", "prop_sib_in_trio", "prob_trio_in_sib")
ukb_r = melt(ukb_overlaps, id.vars = "afcutoff")
names(ukb_r) = c("afcutoff", "category", "result")
ggplot(data = ukb_r , aes(x = afcutoff, y = result, color = category)) + 
geom_point(size=3, shape=17) + theme_bw(base_size=12) + 
scale_x_log10(breaks = c(0.00001,0.0001, 0.001, 0.01, 0.1,1)) +
scale_y_continuous(breaks = seq(0.75, 1, by = 0.05), limits = c(0.75, 1))

In [ ]:
# apply af filter to aou sibling differences and save 
sib_difs_af = sib_difs_af %>% filter(af <= 1e-3)
fwrite(sib_difs_af, "./sibs_snps_QC_GIAB_distance_AF.txt", sep = "\t", quote = FALSE, row.names = FALSE)

In [ ]:
# generate and write final counts 
snps_filtered = sib_difs_af 

sib_counts = data.frame(idv1 = sibs_all$idv1, idv2 = sibs_all$idv2, 
                        par1 = sibs_all$par1, par2 = sibs_all$par2, idv1_count = NA, idv2_count=NA)
for (i in 1:nrow(sib_counts)){
    sib_counts$idv1_count[i] = nrow(snps_filtered %>% filter(idv1 == sib_counts$idv1[i] & 
                                                             idv2 == sib_counts$idv2[i] &
                                                             idv == sib_counts$idv1[i]))
    sib_counts$idv2_count[i] = nrow(snps_filtered %>% filter(idv1 == sib_counts$idv1[i] & 
                                                             idv2 == sib_counts$idv2[i] &
                                                             idv == sib_counts$idv2[i]))
}
                                                             
# read in age_sex 
sib_counts$dob_idv1 = NA
sib_counts$dob_idv2 = NA
sib_counts$dif_age = NA
sib_counts$dif_count = NA
age_sex = fread("./king_files/age_sex.txt", sep = ",", header=TRUE)
for (i in 1:nrow(sib_counts)){
    sib_counts$dob_idv1[i] = as.numeric(substr(age_sex$date_of_birth[which(age_sex$person_id == sib_counts$idv1[i])], 1, 4))
    sib_counts$dob_idv2[i] = as.numeric(substr(age_sex$date_of_birth[which(age_sex$person_id == sib_counts$idv2[i])], 1, 4))
    # idv1 is older sibling 
    if (sib_counts$dob_idv1[i] <= sib_counts$dob_idv2[i]){
        sib_counts$dif_age[i] = sib_counts$dob_idv2[i] - sib_counts$dob_idv1[i]
        sib_counts$dif_count[i] = sib_counts$idv2_count[i] - sib_counts$idv1_count[i]
    } else {
        sib_counts$dif_age[i] = sib_counts$dob_idv1[i] - sib_counts$dob_idv2[i]
        sib_counts$dif_count[i] = sib_counts$idv1_count[i] - sib_counts$idv2_count[i]
    }
                                                          
}
                                                             
sib_counts$qc = NA
dif_ages = unique(sib_counts$dif_age)
for (d in dif_ages){
    sub_idx = which(sib_counts$dif_age == d)
    sub = sib_counts[sub_idx,]
    std = sqrt(mean(sub$idv1_count) + mean(sub$idv2_count))
    mn = mean(sub$dif_count)
    lower = mn - 3*std 
    upper = mn + 3*std
    sib_counts$qc[sub_idx] = sub$dif_count <= upper & sub$dif_count >= lower
}
                                                             
sib_counts_qc = sib_counts %>% filter(qc)

# write qc filtered sib difs                                                              
fwrite(sib_counts_qc, "./relatedness_dec20/passing_sibs_counts_after_qc.txt", sep = "\t", quote = FALSE, 
      row.names = FALSE)                                                             
passing_pairs = paste(sib_counts_qc$idv1, sib_counts_qc$idv2, sep = "_")
snps_filtered$pair = paste(snps_filtered$idv1, snps_filtered$idv2, sep = "_")
snps_filtered = snps_filtered %>% filter(pair %in% passing_pairs)
fwrite(snps_filtered, "./sibs_snps_QC_GIAB_distance_AF_outliers.txt", sep = "\t", quote = FALSE, row.names = FALSE)                                                              
                                                             

In [ ]:
# write final counts and parental ages of birth for trios
trios = fread("./relatedness_dec20/trios_final.txt")
trio_counts = data.frame(par1 = trios$par1, par2 = trios$par2, off = trios$off, num_difs = NA, father_age = NA, mother_age = NA)
for (i in 1:nrow(trios)){
    trios_pair = trio_difs %>% filter(par1 == trio_counts$par1[i] & par2 == trio_counts$par2[i] & off == trio_counts$off[i])
    trio_counts$num_difs[i] = nrow(trios_pair)
}
# read in age_sex 
trio_counts$dob_par1 = NA
trio_counts$dob_par2 = NA
trio_counts$dob_off = NA
trio_counts$sex_par1 = NA
trio_counts$sex_par2 = NA
for (i in 1:nrow(trio_counts)){
    trio_counts$dob_par1[i] = as.numeric(substr(age_sex$date_of_birth[which(age_sex$person_id == trio_counts$par1[i])], 1, 4))
    trio_counts$dob_par2[i] = as.numeric(substr(age_sex$date_of_birth[which(age_sex$person_id == trio_counts$par2[i])], 1, 4))
    trio_counts$dob_off[i] = as.numeric(substr(age_sex$date_of_birth[which(age_sex$person_id == trio_counts$off[i])], 1, 4))
    trio_counts$sex_par1[i] = age_sex$sex_at_birth[which(age_sex$person_id == trio_counts$par1[i])]
    trio_counts$sex_par2[i] = age_sex$sex_at_birth[which(age_sex$person_id == trio_counts$par2[i])]   
}
trio_counts = trio_counts %>% filter(sex_par1 %in% c("Male", "Female") | 
                                     sex_par2 %in% c("Male", "Female"))
nrow(trio_counts)
trio_counts$sex_par1 = ifelse(trio_counts$sex_par2 == "Female", 
                             "Male", "Female")
trio_counts$sex_par2 = ifelse(trio_counts$sex_par1 == "Female", 
                             "Male", "Female")
trio_counts$father_age = ifelse(trio_counts$sex_par1 == "Male",
                               trio_counts$dob_off - trio_counts$dob_par1,
                               trio_counts$dob_off - trio_counts$dob_par2)
trio_counts$mother_age = ifelse(trio_counts$sex_par1 == "Female",
                               trio_counts$dob_off - trio_counts$dob_par1,
                               trio_counts$dob_off - trio_counts$dob_par2)
fwrite(trio_counts, "./relatedness_dec20/trio_counts.txt", sep = "\t", row.names = FALSE, quote = FALSE)

In [ ]:
# write final counts and parental ages of birth for filter-passing siblings 
sibs_age = sib_counts_qc

sib_idv_par = data.frame(off = NA, par1 = NA, par2 = NA, off_dob = NA, par1_dob = NA, 
                        par2_dob = NA, sex_par1 = NA, sex_par2 = NA, fid = NA)
for (i in 1:nrow(sibs_age)){
    idvs = c(sibs_age$idv1[i], sibs_age$idv2[i])
    p1 = sibs_age$par1[i]
    p2 = sibs_age$par2[i]
    p1_dob = NA
    p1_sex = NA
    p2_dob = NA
    p2_sex = NA
    idv1_dob = as.numeric(substr(age_sex$date_of_birth[which(age_sex$person_id == idvs[1])], 1, 4))
    idv2_dob = as.numeric(substr(age_sex$date_of_birth[which(age_sex$person_id == idvs[2])], 1, 4))
    if (!is.na(p1)){
        p1_dob = as.numeric(substr(age_sex$date_of_birth[which(age_sex$person_id == p1)], 1, 4))
        p1_sex = age_sex$sex_at_birth[which(age_sex$person_id == p1)]
    }
    if (!is.na(p2)){
        p2_dob = as.numeric(substr(age_sex$date_of_birth[which(age_sex$person_id == p2)], 1, 4))
        p2_sex = age_sex$sex_at_birth[which(age_sex$person_id == p2)]
    } 
    add = data.frame(off = idvs, par1 = p1, par2 = p2, off_dob = c(idv1_dob, idv2_dob), par1_dob = p1_dob, 
                        par2_dob = p2_dob, sex_par1 = p1_sex, sex_par2 = p2_sex, fid = sibs_age$fid[i])
    sib_idv_par = rbind(sib_idv_par, add)
}
sib_idv_par = sib_idv_par[2:nrow(sib_idv_par),]
sib_idv_par_info = sib_idv_par %>% filter(!is.na(par1) | !is.na(par2))
sib_idv_par_info = unique(sib_idv_par_info)

sib_idv_par_info$mother_dob = ifelse(sib_idv_par_info$sex_par1 == "Female", 
                                    sib_idv_par_info$par1_dob, sib_idv_par_info$par2_dob)
sib_idv_par_info$father_dob = ifelse(sib_idv_par_info$sex_par1 == "Male", 
                                    sib_idv_par_info$par1_dob, sib_idv_par_info$par2_dob)
sib_idv_par_info$skip = (sib_idv_par_info$sex_par1 == 'PMI: Skip' | is.na(sib_idv_par_info$sex_par1)) & 
                        (sib_idv_par_info$sex_par2 == 'PMI: Skip' | is.na(sib_idv_par_info$sex_par2))
sib_idv_par_info = sib_idv_par_info  %>% filter(!skip)
sib_idv_par_info$mother_age = sib_idv_par_info$off_dob - sib_idv_par_info$mother_dob
sib_idv_par_info$father_age = sib_idv_par_info$off_dob - sib_idv_par_info$father_dob
sib_idv_par_info$mother_age[which(sib_idv_par_info$mother_age <= 10)] = NA
sib_idv_par_info$father_age[which(sib_idv_par_info$father_age <= 10)] = NA
fwrite(sib_idv_par_info, "./relatedness_dec20/sibs_passing_qc_with_parental_ages.txt", sep = "\t", 
      row.names = FALSE, quote = FALSE)

In [ ]:
# write ibd2 file to use in 22_remove_giab_from_ibd2_python
# start with aou
ibd2_0.75cm = fread("./ibd2_all_trimmed_0.75cM_jan12.txt")
ibd2_0.75cm$pair = paste(ibd2_0.75cm$ID1, ibd2_0.75cm$ID2, sep = "_")
ibd2_all = ibd2_0.75cm %>% filter(length_cm >= 6)
ibd2_all = unique(ibd2_all)
ibd2_all$length = ibd2_all$new_end_bp-ibd2_all$new_start_bp + 1
ibd2_bed = data.frame(chrom = ibd2_all$Chr, start = ibd2_all$new_start_bp, end = ibd2_all$new_end_bp,
                     pair = ibd2_all$pair)
fwrite(ibd2_bed, "./aou_ibd2_to_remove.bed", row.names= FALSE, col.names = FALSE, quote = FALSE, 
      sep = "\t")

In [ ]:
ibd2_ukb = fread("./ukb/sibs_ibd2_updated.csv",sep = ",")
ibd2_ukb$pair = paste(ibd2_ukb$Sib1, ibd2_ukb$Sib2, sep = "_")
ibd2_bed = data.frame(chrom = ibd2_ukb$Chrom, start =ibd2_ukb$Start, end = ibd2_ukb$End,
                     pair = ibd2_ukb$pair)
fwrite(ibd2_bed, "./ukb_ibd2_to_remove.bed", row.names= FALSE, col.names = FALSE, quote = FALSE, 
      sep = "\t")